# Testing the Prediction Module

This notebook demonstrates how to use the enhanced prediction module that includes features for:
- Skipping existing predictions
- Selecting specific target IDs for prediction
- Forcing or ignoring covariates in predictions

We'll also visualize and analyze the prediction results.

## Setup

First, let's import the necessary libraries and set up our paths:

In [1]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from IPython.display import display, HTML

# Add the parent directory to the path to import our module
# Adjust this if your module is in a different location
module_path = os.path.abspath('..')
if module_path not in sys.path:
    sys.path.append(module_path)

# Import our prediction module
from prediction import make_predictions, load_model
from data_loader import load_data

# Set up paths - modify these to match your environment
DATA_DIR = '../data'
MODELS_DIR = './output_20250307/models'
OUTPUT_DIR = './output_20250307/test_predictions'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_palette('viridis')

## Exploring Available Models

Let's see what models are available in our models directory:

In [2]:
# Find all model files
model_files = glob.glob(os.path.join(MODELS_DIR, '*_model.pkl'))
target_ids = [os.path.basename(f).split('_model.pkl')[0] for f in model_files]

print(f"Found {len(model_files)} model files:")
for i, target_id in enumerate(target_ids):
    # Load model to get type (regression/classification)
    model_data = load_model(model_files[i])
    if model_data:
        target_type = model_data.get('target_type', 'unknown')
        uses_covariates = model_data.get('include_covariates', False)
        print(f"{i+1}. {target_id} - Type: {target_type}, Uses covariates: {uses_covariates}")
    else:
        print(f"{i+1}. {target_id} - Could not load model data")

# Create separate lists for regression and classification targets
regression_targets = []
classification_targets = []

for file in model_files:
    target_id = os.path.basename(file).split('_model.pkl')[0]
    model_data = load_model(file)
    if model_data:
        if model_data.get('target_type') == 'continuous':
            regression_targets.append(target_id)
        elif model_data.get('target_type') == 'discrete':
            classification_targets.append(target_id)

print(f"\nRegression targets: {len(regression_targets)}")
print(f"Classification targets: {len(classification_targets)}")

Found 130 model files:
1. T0027 - Type: discrete, Uses covariates: False
2. T0039 - Type: discrete, Uses covariates: False
3. T0013 - Type: continuous, Uses covariates: False
4. T0577 - Type: continuous, Uses covariates: False
5. T0023 - Type: discrete, Uses covariates: False
6. T0589 - Type: continuous, Uses covariates: False
7. T0562 - Type: continuous, Uses covariates: False
8. T0568 - Type: continuous, Uses covariates: False
9. T0559 - Type: continuous, Uses covariates: False
10. T0031 - Type: discrete, Uses covariates: False
11. T0569 - Type: continuous, Uses covariates: False
12. T0573 - Type: continuous, Uses covariates: False
13. T0552 - Type: continuous, Uses covariates: False
14. T0032 - Type: discrete, Uses covariates: False
15. T0045 - Type: discrete, Uses covariates: False
16. T0019 - Type: continuous, Uses covariates: False
17. T0033 - Type: discrete, Uses covariates: False
18. T0016 - Type: continuous, Uses covariates: False
19. T0584 - Type: continuous, Uses covariates:

## Test 1: Generate Predictions for All Models

Let's first try to generate predictions for all models, with the skip_existing feature enabled:

## Test 2: Force Regenerating Predictions for Specific Targets

Now let's test regenerating predictions for specific targets, even if they already exist:

In [3]:
# Select a subset of targets (adjust based on your available models)
# Try to include at least one regression and one classification target
selected_targets = []

# Add some regression targets if available
if regression_targets:
    selected_targets.extend(regression_targets[:min(2, len(regression_targets))])
    
# Add some classification targets if available
if classification_targets:
    selected_targets.extend(classification_targets[:min(2, len(classification_targets))])

# If we still don't have any targets, use the first few from all targets
if not selected_targets and target_ids:
    selected_targets = target_ids[:min(4, len(target_ids))]

print(f"Selected targets for regeneration: {selected_targets}")

# Force regeneration for selected targets
make_predictions(
    data_dir=DATA_DIR,
    models_dir=MODELS_DIR,
    output_dir=OUTPUT_DIR,
    skip_existing=False,  # Don't skip existing predictions
    target_ids=selected_targets
)

Selected targets for regeneration: ['T0013', 'T0577', 'T0027', 'T0039']
Found 4 of 4 requested model files
Loading data with include_covariates=False
Loaded X data: (1611, 7287) features, 1611 samples
Loaded predictor ID data: (7287, 1)
Loaded Y data: 625 targets, 1611 samples
Sample ID verification successful
Model T0013 uses_covariates=False, prediction uses_covariates=False
Model uses 500 predictors and 0 covariates
Changed device from cuda:0 to cuda:0
Saved predictions for T0013
Model T0577 uses_covariates=False, prediction uses_covariates=False
Model uses 500 predictors and 0 covariates
Changed device from cuda:0 to cuda:0
Saved predictions for T0577
Model T0027 uses_covariates=False, prediction uses_covariates=False
Model uses 500 predictors and 0 covariates
Changed device from cuda:0 to cuda:0
Saved predictions for T0027
Model T0039 uses_covariates=False, prediction uses_covariates=False
Model uses 500 predictors and 0 covariates
Changed device from cuda:0 to cuda:0
Saved predic

## Test 3: Trying with Non-existent Target IDs

Let's see how the function handles non-existent target IDs:

In [4]:
# Try with a mix of valid and invalid target IDs
mixed_targets = []

# Add one valid target if available
if target_ids:
    mixed_targets.append(target_ids[0])
    
# Add some non-existent targets
mixed_targets.extend(['nonexistent_target_1', 'fake_target_2'])

print(f"Mixed targets (valid and invalid): {mixed_targets}")

# Run prediction with mixed targets
make_predictions(
    data_dir=DATA_DIR,
    models_dir=MODELS_DIR,
    output_dir=OUTPUT_DIR,
    target_ids=mixed_targets
)

Mixed targets (valid and invalid): ['T0027', 'nonexistent_target_1', 'fake_target_2']
Found 1 of 3 requested model files
Skipping 1 models with existing predictions: T0027
All predictions already exist. Nothing to do.


## Summary

In this notebook, we've tested the enhanced prediction module with various features:

1. **Basic functionality** - Generating predictions for all models
2. **Selective regeneration** - Forcing predictions for specific target IDs
3. **Error handling** - Testing with non-existent target IDs
4. **Covariate handling** - Testing different covariate settings
5. **Skipping existing predictions** - Verifying file timestamps

The module successfully handles all these scenarios, providing a flexible and efficient way to generate predictions from trained models.